# Guide: metrics

[`pyEDW.metrics`](#pyEDW.metrics) turns an endpoint ensemble into viability and the information measures of Section 4 of the paper{footcite}`sowinski2025exo`.
At each luminosity it histograms the joint distribution $p_{AE}(f_B, f_W, T, L)$, tabulates the Shannon entropies{footcite}`shannon1948` of every marginal, and combines them.

In [ ]:
# Install pyEDW first when this notebook runs somewhere it is not installed,
# such as Google Colab. Where pyEDW is already installed this does nothing.
try:
    import pyEDW
except ImportError:
    %pip install --quiet git+https://github.com/EternalTime/pyEDW.git

In [ ]:
# This page reads the two ensembles that Getting started builds: bio, with
# daisies, and env, its agent-free twin.
import numpy as np
from pyEDW import Parameters, ExoDaisyWorld

p   = Parameters(dT=30.0)
Ls  = 1.0 + np.linspace(-0.7, 1.4, 400)
bio = ExoDaisyWorld.ensemble(p, Ls=Ls, nsteps=2000, N=500, seed=1)
env = ExoDaisyWorld.ensemble(p, Ls=Ls, nsteps=200, N=500,
                             agent_free=True, seed=2)

## Entropy table

In [ ]:
from pyEDW import metrics

H  = metrics.entropy_table(bio)       # (n_lum, 16)
He = metrics.entropy_table_env(env)   # (n_lum, 4), agent-free

Each row of `H` holds the fifteen marginal entropies of $p_{AE}$ plus the bin count, with columns named in the module (`metrics.H_A1A2E1E2` and friends) so the derived measures read like their equations.
Binning uses the square-root rule $N_{bins} = 1 + \lceil\sqrt{n}\rceil$ over the living instances ($f_B f_W > 10^{-7}$); luminosities with fewer than two living instances are `NaN`.

## Derived measures

In [ ]:
V   = metrics.viability(bio, p.f)        # Eq. 9
E   = metrics.efficacy(bio)              # <T> - 1
IAE = metrics.mutual_information(H)       # I(A:E)
dI  = metrics.delta_I(H, He)              # Eq. 12
C   = metrics.cooperation(H)              # Eq. 14

All return one value per luminosity.
[`mutual_information()`](#pyEDW.metrics.mutual_information), [`cooperation()`](#pyEDW.metrics.cooperation), and [`intra_environment_information()`](#pyEDW.metrics.intra_environment_information) also take a table stacked over a 2-D sweep, shape `(n_bandwidth, n_lum, 16)`, and broadcast over the leading axis, so a full bandwidth $\times$ luminosity sweep is a single call.

## Interpreting the signs

$I(A{:}E) \ge 0$ always.
$\Delta I > 0$ means the biome tightens the temperature--luminosity correlation; $\Delta I < 0$ means it loosens it.
Negative $C(a_1{:}a_2\|E)$ means the environment synergistically enhances the correlation between the two daisy species.

### References

```{footbibliography}
```